# 00 — Math foundations

This is the only math notebook. Everything that follows assumes you understand the four ideas here:

1. **Vectors and matrices** — a CNN is matrix multiplications glued together.
2. **Dot product** — every neuron computes one of these.
3. **Derivatives and the chain rule** — backpropagation *is* the chain rule.
4. **Gradient** — the direction "downhill" in a loss landscape.

We'll keep it surgical: introduce a concept, write it in NumPy, see it work.


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. Vectors

A vector is just an ordered list of numbers. Geometrically, it's an arrow from the origin.

$$\vec{v} = \begin{bmatrix} 2 \\ 3 \end{bmatrix}$$

In NumPy:


In [ ]:
v = np.array([2.0, 3.0])
print("vector:", v)
print("length (Euclidean):", np.linalg.norm(v))  # √(2² + 3²) = √13 ≈ 3.606


## 2. Dot product — the atom of neural nets

The dot product of two vectors of the same length is

$$\vec{a} \cdot \vec{b} = \sum_i a_i b_i$$

Geometrically: how aligned the two arrows are. **A single neuron computes a dot product of weights and inputs.**

```
neuron output (pre-activation) = w · x + b
                                  ^^^^^
                                  dot product
```


In [ ]:
w = np.array([0.5, -1.0, 2.0])  # neuron's weights
x = np.array([1.0,  2.0, 0.5])  # an input
b = 0.1                          # bias
pre_activation = w @ x + b       # `@` is matmul; for 1-D it's dot product
print("w · x + b =", pre_activation)


## 3. Matrices and matrix multiplication

A matrix is a 2-D grid of numbers. **A whole layer of neurons is one matrix multiplication.**

If you have a layer with $n$ inputs and $m$ output neurons, its weight matrix is shape $(m, n)$. For one input vector $x$ of shape $(n,)$:

$$h = W \cdot x + \vec{b}$$

`h` has shape $(m,)$ — one number per neuron.


In [ ]:
W = np.array([[ 0.5, -1.0,  2.0],   # neuron 0
              [ 1.0,  0.0,  1.5],   # neuron 1
              [-0.2,  0.3,  0.8]])  # neuron 2
b = np.array([0.1, 0.0, -0.5])
x = np.array([1.0, 2.0, 0.5])
h = W @ x + b
print("layer output:", h)


## 4. Derivatives

A derivative tells you how fast a function changes. For $f(x) = x^2$, the derivative is $f'(x) = 2x$.

Why neural nets care: the **loss** is a function of all the weights. To reduce the loss, we need to know "if I nudge this weight a little, does the loss go up or down?" That's a derivative.


In [ ]:
# Numerically verify d/dx (x^2) = 2x at x=3 → 6.
def f(x): return x**2
x = 3.0
h = 1e-6
slope = (f(x+h) - f(x-h)) / (2*h)
print(f"numerical derivative at x=3: {slope:.6f}   (exact: 6)")


## 5. Chain rule — the entire reason backprop works

If $y = g(f(x))$ then

$$\frac{dy}{dx} = \frac{dy}{df} \cdot \frac{df}{dx}$$

A neural net is a long composition: $\text{loss}(\text{layer}_N(\text{layer}_{N-1}(\dots(\text{layer}_1(x)))))$. To compute the derivative of the loss with respect to any weight, you multiply the local derivatives along the chain. That is backprop. There is no other magic.


In [ ]:
# Example: y = (3x + 1)^2.  dy/dx = 2(3x+1) · 3 = 6(3x+1)
# At x=2: dy/dx = 6 · 7 = 42.
def g(x): return 3*x + 1
def f(g): return g**2
x = 2.0; h = 1e-6
print("numerical:", (f(g(x+h)) - f(g(x-h))) / (2*h), "  exact: 42")


## 6. Gradient — derivative for many variables at once

If your loss depends on a million weights, the **gradient** is just the vector of all those partial derivatives. NumPy lets us compute these efficiently because matrix multiplication and broadcasting handle the bookkeeping.

We're done with prerequisites. Next: load real images, then build a neuron from these four ideas.
